# LSTM-based Reduced-Time Recursive Physics-Informed Neural Network for Shape Memory Materials

## Central-Hole Tension Plate with Sparse Data Learning (LSTM Version)

This notebook implements a physics-informed neural network for thermo-viscoelastic shape memory materials, used to predict the shape memory recovery behavior of 45° fiber-reinforced composite plate with central circular hole.

### Problem Setup

- **Geometry**: Square plate with central hole (20mm × 20mm × 1mm, hole radius 3mm at center (10,10))
- **Boundary Conditions**: Left face (x=0) fixed via RP1, right face (x=20mm) with prescribed displacement via RP2
- **Loading History**: Shape memory cycle (programming → cooling → unloading → reheating)
- **Data Strategy**: Sparse sampling (Enhanced: 30% spatial × 70% temporal = 21% total)
- **Key Feature**: Stress concentration analysis around central hole

## Import Required Libraries

Import libraries for numerical computation, deep learning, data processing, and visualization.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
import time
from pathlib import Path
from matplotlib.tri import Triangulation
from scipy.interpolate import interp1d
import sys

## FEDataLoader

Implementation of the `FEDataLoader` class.

Load and process FE simulation results from CSV files for central-hole plate geometry

In [ ]:
# Import complete implementation from lstm_pinn_ex3.py
from lstm_pinn_ex3 import (
    FEDataLoader,
    MaterialParameters,
    LSTM_PINN,
    LSTMPINNSolver,
    plot_training_history,
    plot_sampling_strategy,
    plot_displacement_field,
    plot_shape_memory_cycle,
    device
)

print("All components imported successfully from lstm_pinn_ex3.py")
print(f"Using device: {device}")

## main

Implementation of the `main` function.

In [ ]:
def main():

    """

    Main training function for LSTM-PINN on central-hole tension plate

    """

    print("=" * 80)

    print("LSTM-PINN for Shape Memory Materials")

    print("Central-Hole Tension Plate (20×20×1 mm, r=3mm)")

    print("=" * 80)

    print()

    

    # Setup paths

    script_dir = Path(__file__).resolve().parent if '__file__' in globals() else Path('.')

    data_dir = script_dir / 'EX-3-RESULTS'

    step_frame_time_file = script_dir / 'step-frame-time.csv'

    

    # Load FE data

    print("Loading FE data...")

    fe_loader = FEDataLoader(data_dir, step_frame_time_file)

    full_data = fe_loader.load_all_data()

    bounds = fe_loader.get_domain_bounds()

    print(f"  Total data points: {len(full_data):,}")

    print(f"  Domain: x∈[{bounds['x_min']:.1f},{bounds['x_max']:.1f}], "

          f"y∈[{bounds['y_min']:.1f},{bounds['y_max']:.1f}], "

          f"z∈[{bounds['z_min']:.1f},{bounds['z_max']:.1f}]")

    print(f"  Time: t∈[{bounds['t_min']:.1f},{bounds['t_max']:.1f}] s")

    print()

    

    # Initialize material parameters

    mat_params = MaterialParameters()

    print("Material parameters initialized.")

    print(f"  Reference temperature: {mat_params.T_ref:.1f} K ({mat_params.T_ref - 273.15:.1f} C)")

    print(f"  Switch temperature: {mat_params.T_switch:.1f} K ({mat_params.T_switch - 273.15:.1f} C)")

    print(f"  Prony series: {mat_params.N_prony} terms")

    print()

    

    # Initialize LSTM-PINN model

    print("Initializing LSTM-PINN...")

    model = LSTM_PINN(

        spatial_layers=[5, 256, 256, 256],  # x, y, z, t, T

        lstm_hidden=512,

        lstm_layers=3,

        output_layers=[256, 128, 3],  # u_x, u_y, u_z

        n_prony=mat_params.N_prony,

        output_internal_vars=True,

        dropout=0.02

    )

    

    print(f"  Total parameters: {model.count_parameters():,}")

    print(f"  Architecture:")

    print(f"    Spatial encoder: [5, 256, 256, 256]")

    print(f"    LSTM: 512 hidden × 3 layers")

    print(f"    Output decoder: [256, 128, 3]")

    print(f"    Internal vars: 6 × {mat_params.N_prony}")

    print()

    

    # Initialize solver

    solver = LSTMPINNSolver(model, fe_loader, mat_params, device=device)

    print()

    

    # Train model

    print("Training phase:")

    print("=" * 80)

    print("PINN Advantage: Learning from Sparse Data")

    print("  Full dataset: ~62M points (278k nodes x 226 frames)")

    print("  Sparse sampling: ~13.2M points (30% spatial x 70% temporal = 21% total)")

    print("  Enhanced configuration:")

    print("    - Loss weights: Data=2.0, BC=2.0, PDE=0.05")

    print("    - Emphasized hole boundary sampling (40% vs 30%)")

    print()

    

    history, sparse_data = solver.train(

        epochs=12000,

        batch_size=512,

        learning_rate=1e-4,

        n_pde_points=1000,

        n_bc_points=200,

        spatial_sampling_ratio=0.3,   # ENHANCED: 30% of nodes

        temporal_sampling_ratio=0.7    # ENHANCED: 70% of frames

    )

    

    # Save model

    print("\
Saving model...")

    torch.save(model.state_dict(), script_dir / 'lstm_pinn_model_ex3.pth')

    history_csv = script_dir / 'lstm_ex3_training_history.csv'
    history_records = []
    for epoch_idx in range(len(history['loss_total'])):
        history_records.append({
            'epoch': epoch_idx + 1,
            'loss_total': history['loss_total'][epoch_idx],
            'loss_data': history['loss_data'][epoch_idx],
            'loss_pde': history['loss_pde'][epoch_idx],
            'loss_bc': history['loss_bc'][epoch_idx],
            'loss_evol': history['loss_evol'][epoch_idx],
        })
    pd.DataFrame(history_records).to_csv(history_csv, index=False)

    print(f"  Model saved to: lstm_pinn_model_ex3.pth")

    print(f"  Training history saved to: {history_csv}")

    

    # Visualizations

    print("\
Generating visualizations...")

    

    # 1. Training history

    solver.loss_history = history['loss_total']
    solver.loss_components_history = []
    for i in range(len(history['loss_total'])):
        solver.loss_components_history.append({
            'data': history['loss_data'][i],
            'pde': history['loss_pde'][i],
            'bc': history['loss_bc'][i],
            'evol': history['loss_evol'][i],
        })
    plot_training_history(solver, save_dir=script_dir, prefix='lstm_')

    print("  ✓ Training history plot saved")

    

    # 2. Sampling strategy

    plot_sampling_strategy(sparse_data, full_data, save_dir=script_dir)

    print("  ✓ Sampling strategy plot saved")

    

    # 3. Displacement field comparison

    plot_displacement_field(solver, fe_loader, save_dir=script_dir)

    print("  ✓ Displacement field comparison saved")

    

    # 4. Shape memory cycle

    plot_shape_memory_cycle(solver, fe_loader, save_dir=script_dir)

    print("  ✓ Shape memory cycle plot saved")

    

    print("\
" + "=" * 80)

    print("LSTM-PINN training and evaluation completed successfully!")

    print("=" * 80)

    print("\
Output files:")

    print("  - lstm_pinn_model_ex3.pth")

    print("  - lstm_ex3_training_history.csv")

    print("  - lstm_ex3_training_history.png")

    print("  - lstm_ex3_sampling_strategy.png")

    print("  - lstm_ex3_displacement_field_2d.png")

    print("  - lstm_ex3_shape_memory_cycle.png")

    print()





if __name__ == "__main__":

    main()
